In [1]:
import numpy as np
import os,re
from PIL import Image


# 图像切割及特征提取
path = '../../demo/data/images/'  # 图片所在路径

In [2]:
# 自定义获取图片名称函数
def getImgNames(path=path):
    '''
    获取指定路径中所有图片的名称
    :param path: 指定的路径
    :return: 名称列表
    '''
    filenames = os.listdir(path)  # 根据路径获取文件名
    imgNames = []
    for i in filenames:
        # 检查文件是否为图片格式（可以根据需要调整扩展名）
        if re.search(r'\.(jpg|jpeg|png|bmp)$', i, re.IGNORECASE):
            imgNames.append(i)
    return imgNames

In [3]:
# 自定义获取三阶颜色矩函数
def Var(data=None):
    '''
    获取给定像素值矩阵的三阶颜色矩
    :param data: 给定的像素值矩阵
    :return: 对应的三阶颜色矩
    '''
    if data is None or len(data) == 0:
        return 0
    
    # 计算像素值的均值
    mean_val = np.mean(data)
    # 计算每个像素值与均值的差的三次方的平均值
    x = np.mean((data - mean_val) ** 3)
    # 三阶矩计算 - 使用立方根来保持数值稳定性
    return np.sign(x) * abs(x) ** (1/3)

In [4]:
# 批量处理图片数据
imgNames = getImgNames(path=path)  # 获取所有图片名称
n = len(imgNames)        # 图片张数
data = np.zeros([n, 9])  # 用来装样本自变量
labels = np.zeros([n])   # 用来放样本标签

In [5]:
# 图像切割和特征提取
n = len(imgNames)  # 图片数量
data = np.zeros((n, 9))  # 9个特征：RGB三通道的三个颜色矩
labels = np.zeros(n)     # 标签数组

for i in range(n):
    img = Image.open(path + imgNames[i])  # 读取图片
    M, N = img.size  # 图片像素的尺寸
    
    # 图片切割 - 从中心切割100x100的区域
    img = img.crop((M/2-50, N/2-50, M/2+50, M/2+50))
    
    # 将图片分割成三通道
    r, g, b = img.split()
    
    # 转化成数组数据
    rd = np.array(r, dtype=np.float64).flatten()
    gd = np.array(g, dtype=np.float64).flatten()
    bd = np.array(b, dtype=np.float64).flatten()
    
    # 一阶颜色矩（均值）
    data[i, 0] = np.mean(rd)
    data[i, 1] = np.mean(gd)
    data[i, 2] = np.mean(bd)
    
    # 二阶颜色矩（标准差）
    data[i, 3] = np.std(rd)
    data[i, 4] = np.std(gd)
    data[i, 5] = np.std(bd)
    
    # 三阶颜色矩（偏度）
    data[i, 6] = Var(rd)
    data[i, 7] = Var(gd)
    data[i, 8] = Var(bd)
    
    # 样本标签 - 假设文件名第一个字符是水质等级
    labels[i] = int(imgNames[i][0])

In [6]:
from sklearn.model_selection import train_test_split
# 数据拆分,训练集、测试集
data_tr,data_te,label_tr,label_te = train_test_split(data,labels,test_size=0.2,
                                                     random_state=10)

In [7]:
from sklearn.tree import DecisionTreeClassifier

# 模型训练
model = DecisionTreeClassifier(
    random_state=42,          # 随机种子，确保结果可重现
    max_depth=5,              # 树的最大深度，防止过拟合
    min_samples_split=10,     # 内部节点再划分所需最小样本数
    min_samples_leaf=5,       # 叶子节点最少样本数
    criterion='gini'          # 分裂标准，也可以使用'entropy'
)  # 建立决策树模型

# 训练模型
model.fit(data_tr, label_tr)

,criterion,'gini'
,splitter,'best'
,max_depth,5
,min_samples_split,10
,min_samples_leaf,5
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [8]:
# 水质评价
from sklearn.metrics import confusion_matrix

pre_te = model.predict(data_te)  # 使用训练好的模型对测试集进行预测

In [9]:
pre_te

array([2., 2., 1., 3., 1., 2., 2., 2., 3., 2., 4., 3., 3., 1., 2., 3., 1.,
       2., 3., 1., 3., 2., 1., 3., 3., 3., 1., 2., 2., 2., 2., 2., 1., 3.,
       1., 5., 3., 1., 4., 3., 4.])

In [10]:
# 混淆矩阵
cm_te = confusion_matrix(label_te, pre_te)  # 计算混淆矩阵
print("混淆矩阵:")
print(cm_te)

混淆矩阵:
[[ 4  4  3  0  0]
 [ 2  9  0  0  0]
 [ 4  0 10  0  0]
 [ 0  1  0  3  1]
 [ 0  0  0  0  0]]


In [11]:
from sklearn.metrics import accuracy_score

# 准确率
print("测试集准确率:", accuracy_score(label_te, pre_te))  # 计算准确率

测试集准确率: 0.6341463414634146


In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils import resample
import time
import warnings
warnings.filterwarnings('ignore')

# 加载实际数据
def load_environment_data(file_path):
    """
    加载环境监测数据
    """
    print("正在加载环境监测数据...")
    try:
        # 读取Excel文件
        df = pd.read_excel(file_path)
        print(f"数据加载成功！数据形状: {df.shape}")
        print(f"数据列名: {df.columns.tolist()}")
        print("\n数据前5行:")
        print(df.head())
        
        return df
    except Exception as e:
        print(f"数据加载失败: {e}")
        return None

# 数据预处理
def preprocess_data(df):
    """
    数据预处理
    """
    print("\n" + "="*50)
    print("数据预处理")
    print("="*50)
    
    # 检查缺失值
    missing_values = df.isnull().sum()
    print("缺失值统计:")
    print(missing_values[missing_values > 0])
    
    # 处理缺失值
    df_clean = df.copy()
    
    # 对于数值列，用中位数填充
    numeric_columns = df_clean.select_dtypes(include=[np.number]).columns
    for col in numeric_columns:
        if df_clean[col].isnull().sum() > 0:
            df_clean[col].fillna(df_clean[col].median(), inplace=True)
    
    # 对于分类列，用众数填充
    categorical_columns = df_clean.select_dtypes(include=['object']).columns
    for col in categorical_columns:
        if df_clean[col].isnull().sum() > 0:
            df_clean[col].fillna(df_clean[col].mode()[0], inplace=True)
    
    print("缺失值处理完成！")
    
    return df_clean

# 检查数据平衡性并处理
def check_and_balance_data(df, target_column):
    """
    检查目标变量的分布并进行平衡处理
    """
    print("\n目标变量分布检查:")
    target_distribution = df[target_column].value_counts().sort_index()
    print(target_distribution)
    
    # 检查是否有样本数少于2的类别
    insufficient_classes = target_distribution[target_distribution < 2].index.tolist()
    
    if insufficient_classes:
        print(f"\n警告: 以下类别的样本数少于2个: {insufficient_classes}")
        print("这些类别将被移除...")
        
        # 移除样本数不足的类别
        df_balanced = df[~df[target_column].isin(insufficient_classes)].copy()
        
        print(f"移除后数据形状: {df_balanced.shape}")
        print("新的目标变量分布:")
        print(df_balanced[target_column].value_counts().sort_index())
    else:
        df_balanced = df.copy()
        print("所有类别的样本数都足够")
    
    return df_balanced

# 特征工程和数据准备
def prepare_features_target(df, target_column=None):
    """
    准备特征和目标变量
    """
    print("\n" + "="*50)
    print("特征工程")
    print("="*50)
    
    if target_column is None:
        # 常见的环境质量标签列名
        possible_targets = ['水质等级', '水质类别', '质量等级', '类别', '等级', 'label', 'target', 'quality', 
                           '水质', '评价等级', '水质评价', '水质量等级']
        
        for col in possible_targets:
            if col in df.columns:
                target_column = col
                break
        
        if target_column is None:
            target_column = df.columns[-1]
            print(f"未找到明确的目标列，使用最后一列: {target_column}")
    
    print(f"目标变量: {target_column}")
    
    # 检查并平衡数据
    df_balanced = check_and_balance_data(df, target_column)
    
    # 分离特征和目标
    X = df_balanced.drop(columns=[target_column])
    y = df_balanced[target_column]
    
    # 处理分类特征
    categorical_columns = X.select_dtypes(include=['object']).columns
    numeric_columns = X.select_dtypes(include=[np.number]).columns
    
    print(f"分类特征: {categorical_columns.tolist()}")
    print(f"数值特征: {numeric_columns.tolist()}")
    
    # 对分类特征进行编码
    label_encoders = {}
    X_encoded = X.copy()
    
    for col in categorical_columns:
        le = LabelEncoder()
        X_encoded[col] = le.fit_transform(X[col].astype(str))
        label_encoders[col] = le
        print(f"编码分类特征 '{col}': {dict(zip(le.classes_, le.transform(le.classes_)))}")
    
    # 对目标变量进行编码（如果是分类的）
    if y.dtype == 'object':
        le_target = LabelEncoder()
        y_encoded = le_target.fit_transform(y)
        print(f"目标变量编码: {dict(zip(le_target.classes_, le_target.transform(le_target.classes_)))}")
    else:
        y_encoded = y.values
        # 如果目标变量是数值型，检查是否需要转换为分类
        unique_values = np.unique(y_encoded)
        if len(unique_values) <= 10:  # 如果唯一值少于10个，认为是分类问题
            print(f"目标变量数值转换为分类，类别: {unique_values}")
        else:
            print(f"目标变量有 {len(unique_values)} 个唯一值，可能是回归问题")
    
    print(f"最终特征形状: {X_encoded.shape}")
    print(f"目标变量分布: {pd.Series(y_encoded).value_counts().sort_index()}")
    
    return X_encoded, y_encoded, X_encoded.columns.tolist(), label_encoders, le_target if 'le_target' in locals() else None

# 安全的数据划分函数
def safe_train_test_split(X, y, test_size=0.2, random_state=42):
    """
    安全的数据划分，处理样本数少的类别
    """
    # 检查每个类别的样本数
    unique, counts = np.unique(y, return_counts=True)
    min_samples = counts.min()
    
    print(f"\n数据划分信息:")
    print(f"总样本数: {len(y)}")
    print(f"类别数: {len(unique)}")
    print(f"最小类别样本数: {min_samples}")
    
    # 如果最小样本数太少，不使用分层抽样
    if min_samples < 2:
        print("警告: 有类别的样本数少于2，使用随机划分而不是分层划分")
        return train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=None)
    else:
        # 确保测试集中每个类别至少有1个样本
        test_samples_per_class = max(1, int(test_size * min_samples))
        if test_samples_per_class < 1:
            test_size_adjusted = 1 / min_samples
            print(f"调整测试集比例至: {test_size_adjusted:.3f}")
            return train_test_split(X, y, test_size=test_size_adjusted, random_state=random_state, stratify=y)
        else:
            return train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=y)

# C4.5算法
def build_c45_model(data_tr, label_tr):
    """
    构建C4.5决策树模型
    """
    print("=" * 50)
    print("C4.5算法模型训练")
    print("=" * 50)
    
    start_time = time.time()
    
    model_c45 = DecisionTreeClassifier(
        random_state=42,
        criterion='entropy',
        max_depth=8,
        min_samples_split=2,  
        min_samples_leaf=1,
        max_features='sqrt'
    )
    
    model_c45.fit(data_tr, label_tr)
    training_time = time.time() - start_time
    
    print("C4.5模型训练完成！")
    print(f"训练时间: {training_time:.4f}秒")
    
    return model_c45

# CART算法
def build_cart_model(data_tr, label_tr):
    """
    构建CART决策树模型
    """
    print("=" * 50)
    print("CART算法模型训练")
    print("=" * 50)
    
    start_time = time.time()
    
    model_cart = DecisionTreeClassifier(
        random_state=42,
        criterion='gini',
        max_depth=8,
        min_samples_split=2,  # 减小参数以适应小数据集
        min_samples_leaf=1,
        max_features='sqrt'
    )
    
    model_cart.fit(data_tr, label_tr)
    training_time = time.time() - start_time
    
    print("CART模型训练完成！")
    print(f"训练时间: {training_time:.4f}秒")
    
    return model_cart

# 模型评估函数
def evaluate_model(model, data_tr, label_tr, data_te, label_te, model_name, feature_names=None):
    """
    全面评估模型性能
    """
    print(f"\n{model_name} 模型评估")
    print("-" * 40)
    
    # 预测
    start_time = time.time()
    pred_tr = model.predict(data_tr)
    pred_te = model.predict(data_te)
    prediction_time = time.time() - start_time
    
    # 计算各项指标
    accuracy_tr = accuracy_score(label_tr, pred_tr)
    accuracy_te = accuracy_score(label_te, pred_te)
    
    # 检查测试集中是否有多个类别
    unique_test = np.unique(label_te)
    if len(unique_test) > 1:
        precision_te = precision_score(label_te, pred_te, average='weighted', zero_division=0)
        recall_te = recall_score(label_te, pred_te, average='weighted', zero_division=0)
        f1_te = f1_score(label_te, pred_te, average='weighted', zero_division=0)
    else:
        precision_te = recall_te = f1_te = accuracy_te
        print("警告: 测试集中只有一个类别，部分指标与准确率相同")
    
    print(f"训练集准确率: {accuracy_tr:.4f}")
    print(f"测试集准确率: {accuracy_te:.4f}")
    print(f"测试集精确率: {precision_te:.4f}")
    print(f"测试集召回率: {recall_te:.4f}")
    print(f"测试集F1分数: {f1_te:.4f}")
    print(f"预测时间: {prediction_time:.4f}秒")
    
    # 树的结构信息
    n_nodes = model.tree_.node_count
    n_leaves = model.tree_.n_leaves
    depth = model.get_depth()
    
    print(f"树结构 - 节点数: {n_nodes}, 叶节点数: {n_leaves}, 深度: {depth}")
    
    # 特征重要性
    if feature_names is not None:
        print("\n特征重要性排序:")
        feature_importance = pd.DataFrame({
            'feature': feature_names,
            'importance': model.feature_importances_
        }).sort_values('importance', ascending=False)
        
        for _, row in feature_importance.head(10).iterrows():
            print(f"  {row['feature']}: {row['importance']:.4f}")
    
    return {
        'model_name': model_name,
        'accuracy_train': accuracy_tr,
        'accuracy_test': accuracy_te,
        'precision': precision_te,
        'recall': recall_te,
        'f1_score': f1_te,
        'prediction_time': prediction_time,
        'n_nodes': n_nodes,
        'n_leaves': n_leaves,
        'depth': depth
    }

# 主函数
def main():
    """
    主函数：加载数据、预处理、训练和比较模型
    """
    # 文件路径
    file_path = '../data/environment_data.xls'
    
    # 1. 加载数据
    df = load_environment_data(file_path)
    if df is None:
        return
    
    # 2. 数据预处理
    df_clean = preprocess_data(df)
    
    # 3. 准备特征和目标变量
    X, y, feature_names, label_encoders, target_encoder = prepare_features_target(df_clean)
    
    # 4. 安全划分训练集和测试集
    data_tr, data_te, label_tr, label_te = safe_train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    print(f"\n数据划分完成:")
    print(f"训练集: {data_tr.shape}")
    print(f"测试集: {data_te.shape}")
    print(f"训练集类别分布: {pd.Series(label_tr).value_counts().sort_index()}")
    print(f"测试集类别分布: {pd.Series(label_te).value_counts().sort_index()}")
    
    # 5. 训练模型
    model_c45 = build_c45_model(data_tr, label_tr)
    model_cart = build_cart_model(data_tr, label_tr)
    
    # 6. 评估模型
    results = []
    results.append(evaluate_model(model_c45, data_tr, label_tr, data_te, label_te, "C4.5算法", feature_names))
    results.append(evaluate_model(model_cart, data_tr, label_tr, data_te, label_te, "CART算法", feature_names))
    
    # 7. 模型比较
    print("\n" + "=" * 60)
    print("模型性能比较")
    print("=" * 60)
    
    comparison_df = pd.DataFrame(results)
    comparison_display = comparison_df[['model_name', 'accuracy_test', 'precision', 'recall', 'f1_score', 'prediction_time']]
    print(comparison_display.round(4))
    
    # 找出最佳模型
    best_model = comparison_df.loc[comparison_df['f1_score'].idxmax()]
    print(f"\n最佳模型: {best_model['model_name']} (F1分数: {best_model['f1_score']:.4f})")
    
    return model_c45, model_cart, comparison_df, df_clean

# 运行主函数
if __name__ == "__main__":
    model_c45, model_cart, results, processed_data = main()

正在加载环境监测数据...
数据加载成功！数据形状: (320, 7)
数据列名: ['SO2', 'NO', 'NO2', 'NOx', 'PM10', 'PM2-5', '空气等级']

数据前5行:
     SO2   NO    NO2    NOx   PM10  PM2-5 空气等级
0  0.031  0.0  0.046  0.047  0.085  0.058    I
1  0.022  0.0  0.053  0.053  0.070  0.048   II
2  0.017  0.0  0.029  0.029  0.057  0.040    I
3  0.026  0.0  0.026  0.026  0.049  0.034    I
4  0.018  0.0  0.027  0.027  0.051  0.035    I

数据预处理
缺失值统计:
Series([], dtype: int64)
缺失值处理完成！

特征工程
未找到明确的目标列，使用最后一列: 空气等级
目标变量: 空气等级

目标变量分布检查:
空气等级
I       63
II     146
III     83
IV      11
V       10
VI       6
VII      1
Name: count, dtype: int64

警告: 以下类别的样本数少于2个: ['VII']
这些类别将被移除...
移除后数据形状: (319, 7)
新的目标变量分布:
空气等级
I       63
II     146
III     83
IV      11
V       10
VI       6
Name: count, dtype: int64
分类特征: []
数值特征: ['SO2', 'NO', 'NO2', 'NOx', 'PM10', 'PM2-5']
目标变量编码: {'I': 0, 'II': 1, 'III': 2, 'IV': 3, 'V': 4, 'VI': 5}
最终特征形状: (319, 6)
目标变量分布: 0     63
1    146
2     83
3     11
4     10
5      6
Name: count, dtype: int64

数据划分信息:
总样本数: 31